# 02 — Trial Repetition Analysis

Explores variance across repetitions within conditions and potential drift across trials. Directly relevant to **across-repetition CPD** (DATASET_GUIDE §8B): "Does the response distribution drift across repeats?"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else (Path.cwd() / ".." / "..").resolve()
DATA_ROOT = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts" / "figures" / "trial_repetition"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PARTICIPANTS = [f"sub-{i:02d}" for i in range(1, 11)]
DATE_TAG = "2026-02-25"

In [ ]:
def load_preprocessed(path):
    d = np.load(path, allow_pickle=True).item()
    return d["preprocessed_eeg_data"], d["ch_names"], d["times"]

## 1. Within-condition variance across repetitions

For each condition: variance across reps (train: 4 reps, test: 80 reps). Channel-averaged for simplicity.

In [ ]:
rows = []
for sub in PARTICIPANTS:
    p_tr = DATA_ROOT / sub / "preprocessed_eeg_training.npy"
    p_te = DATA_ROOT / sub / "preprocessed_eeg_test.npy"
    if not p_tr.exists() or not p_te.exists():
        continue
    X_tr, ch_names, times = load_preprocessed(p_tr)
    X_te, _, _ = load_preprocessed(p_te)
    # Variance across reps, mean over channels and time: [n_conditions]
    var_tr = X_tr.var(axis=1).mean(axis=(1, 2))
    var_te = X_te.var(axis=1).mean(axis=(1, 2))
    rows.append({
        "participant": sub,
        "train_var_mean": var_tr.mean(),
        "train_var_std": var_tr.std(),
        "test_var_mean": var_te.mean(),
        "test_var_std": var_te.std(),
    })

var_df = pd.DataFrame(rows)
print(var_df.to_string(index=False))

## 2. Variance distribution across participants

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(var_df["participant"], var_df["train_var_mean"], color="steelblue", alpha=0.8)
axes[0].set_xlabel("Participant")
axes[0].set_ylabel("Mean within-condition variance")
axes[0].set_title("Train (4 reps per condition)")
axes[0].tick_params(axis="x", rotation=45)
axes[1].bar(var_df["participant"], var_df["test_var_mean"], color="coral", alpha=0.8)
axes[1].set_xlabel("Participant")
axes[1].set_ylabel("Mean within-condition variance")
axes[1].set_title("Test (80 reps per condition)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / f"trial_rep__variance_by_participant__{DATE_TAG}.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Drift across test repetitions

Split 80 test reps into 8 bins of 10. Mean response per bin per condition. Does it shift across bins?

In [ ]:
N_BINS = 8
REPS_PER_BIN = 80 // N_BINS
drift_rows = []

for sub in PARTICIPANTS:
    p_te = DATA_ROOT / sub / "preprocessed_eeg_test.npy"
    if not p_te.exists():
        continue
    X_te, ch_names, times = load_preprocessed(p_te)
    # X_te: [200, 80, 17, 100]
    bin_means = []
    for b in range(N_BINS):
        start, end = b * REPS_PER_BIN, (b + 1) * REPS_PER_BIN
        m = X_te[:, start:end].mean(axis=(1, 2, 3))
        bin_means.append(m.mean())
    for b, v in enumerate(bin_means):
        drift_rows.append({"participant": sub, "bin": b, "bin_label": f"{b*10+1}-{(b+1)*10}", "mean_amp": v})

drift_df = pd.DataFrame(drift_rows)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for sub in PARTICIPANTS:
    d = drift_df[drift_df["participant"] == sub]
    ax.plot(d["bin"], d["mean_amp"], "o-", alpha=0.7, label=sub)
ax.set_xlabel("Repetition bin (1-10, 11-20, ...)")
ax.set_ylabel("Mean amplitude (channel-avg, condition-avg)")
ax.set_title("Drift across test repetitions (80 reps → 8 bins)")
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / f"trial_rep__drift_across_reps__{DATE_TAG}.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Variance by channel (which channels most variable?)

Per participant: mean within-condition variance per channel (train data).

In [ ]:
channel_var_rows = []
for sub in PARTICIPANTS:
    p_tr = DATA_ROOT / sub / "preprocessed_eeg_training.npy"
    if not p_tr.exists():
        continue
    X_tr, ch_names, times = load_preprocessed(p_tr)
    var_per_ch = X_tr.var(axis=1).mean(axis=(0, 2))
    for i, ch in enumerate(ch_names):
        channel_var_rows.append({"participant": sub, "channel": ch, "var": var_per_ch[i]})

ch_var_df = pd.DataFrame(channel_var_rows)
ch_mean = ch_var_df.groupby("channel")["var"].mean().sort_values(ascending=False)
print("Channels by mean variance (across participants):")
print(ch_mean.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ch_order = ch_mean.index.tolist()
for sub in PARTICIPANTS:
    d = ch_var_df[ch_var_df["participant"] == sub]
    d = d.set_index("channel").reindex(ch_order)["var"]
    ax.plot(range(len(ch_order)), d.values, "o-", alpha=0.6, label=sub)
ax.set_xticks(range(len(ch_order)))
ax.set_xticklabels(ch_order, rotation=45, ha="right")
ax.set_xlabel("Channel")
ax.set_ylabel("Within-condition variance")
ax.set_title("Variance by channel (train, 4 reps)")
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / f"trial_rep__variance_by_channel__{DATE_TAG}.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Save summary table

In [ ]:
tables_dir = ROOT / "artifacts" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
var_df.to_csv(tables_dir / f"trial_rep__variance_summary__{DATE_TAG}.csv", index=False)
print(f"Saved variance summary to artifacts/tables/trial_rep__variance_summary__{DATE_TAG}.csv")